In [1]:
from pathlib import Path
from collections import Counter
import pandas as pd
from PIL import Image

# Ruta del dataset
DATASET_DIR = Path("/kaggle/input/datasets/claudiobances/coffee-leaf-dataset-by-phytosanitary-class/coffee_leaves/")

# Clases que utilizaremos
CLASSES = ["Roya", "Sanas"]

# Extensiones permitidas
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Dataset:", DATASET_DIR)
print("Existe:", DATASET_DIR.exists())

Dataset: /kaggle/input/datasets/claudiobances/coffee-leaf-dataset-by-phytosanitary-class/coffee_leaves
Existe: True


In [2]:
results = []

for class_name in CLASSES:
    class_path = DATASET_DIR / class_name

    images = [
        p for p in class_path.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    ]

    results.append({
        "Clase": class_name,
        "Cantidad de imágenes": len(images)
    })

df = pd.DataFrame(results)

print(df)
print("\nTotal:", df["Cantidad de imágenes"].sum())

   Clase  Cantidad de imágenes
0   Roya                   500
1  Sanas                   500

Total: 1000


In [3]:
for class_name in CLASSES:

    class_path = DATASET_DIR / class_name

    valid = 0
    invalid = []

    for path in class_path.rglob("*"):

        if path.suffix.lower() not in EXTENSIONS:
            continue

        try:
            with Image.open(path) as img:
                img.verify()
            valid += 1

        except Exception:
            invalid.append(str(path))

    print(f"\nClase: {class_name}")
    print(f"Imágenes válidas: {valid}")
    print(f"Imágenes inválidas: {len(invalid)}")

    if invalid:
        print("Archivos problemáticos:")
        print(invalid[:10])


Clase: Roya
Imágenes válidas: 500
Imágenes inválidas: 0

Clase: Sanas
Imágenes válidas: 500
Imágenes inválidas: 0


In [4]:
from pathlib import Path
from collections import defaultdict
import hashlib
import pandas as pd

CLASSES = ["Roya", "Sanas"]
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Agrupar archivos según el hash de su contenido
hash_groups = defaultdict(list)

for class_name in CLASSES:
    class_path = DATASET_DIR / class_name

    for path in class_path.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in EXTENSIONS:
            continue

        sha256 = hashlib.sha256(path.read_bytes()).hexdigest()
        hash_groups[sha256].append({
            "clase": class_name,
            "ruta": str(path)
        })

# Conservar únicamente los grupos con más de una copia
duplicates = {
    sha: files
    for sha, files in hash_groups.items()
    if len(files) > 1
}

print("Grupos de imágenes duplicadas:", len(duplicates))

for sha, files in duplicates.items():
    print(f"\nSHA-256: {sha}")
    for file in files:
        print(f"  [{file['clase']}] {file['ruta']}")

# Resumen para documentar
duplicate_rows = []

for sha, files in duplicates.items():
    for file in files:
        duplicate_rows.append({
            "sha256": sha,
            "clase": file["clase"],
            "ruta": file["ruta"]
        })

duplicates_df = pd.DataFrame(duplicate_rows)

if not duplicates_df.empty:
    duplicates_df.to_csv(
        "duplicados_exactos.csv",
        index=False,
        encoding="utf-8"
    )
    print("\nReporte guardado en duplicados_exactos.csv")
else:
    print("\nNo se encontraron duplicados exactos.")

Grupos de imágenes duplicadas: 0

No se encontraron duplicados exactos.
